In [ ]:
import os
os.chdir('/workspaces/korea-real-estate-population-movement')

from src.config import DB_URL
import pandas as pd
from sqlalchemy import create_engine, inspect

In [ ]:
engine = create_engine(DB_URL)

In [ ]:
inspector = inspect(engine)

for table in inspector.get_table_names():
    print(f"\n--- {table} ---")
    
    for column in inspector.get_columns(table):
        print(column["name"], column["type"])

In [ ]:
df = pd.read_sql(
    f"SELECT * FROM seoul_population_flow LIMIT 5",
    engine
)

display(df)


In [27]:
yearly_top_five_outflow_query = '''
SELECT 
    from_district,
    SUM(total_people) as total_outflow
FROM seoul_population_flow
WHERE TO_DATE(date::TEXT,'YYYYMM') >= TO_DATE('202301','YYYYMM')
AND TO_DATE(date::TEXT,'YYYYMM') <= TO_DATE('202312','YYYYMM')
GROUP BY from_district
ORDER BY total_outflow DESC
LIMIT 5
'''

df = pd.read_sql_query(yearly_top_five_outflow_query,engine)

def create_top_five_str(df,col):
    districts = df[col]
    return "(" + ", ".join(f"'{district}'" for district in districts) + ")"


monthly_outflow_of_top_five_outflow = f'''
SELECT 
    from_district,
    date,
    SUM(total_people) as total_outflow
FROM seoul_population_flow
WHERE TO_DATE(date::TEXT,'YYYYMM') >= TO_DATE('202301','YYYYMM')
AND TO_DATE(date::TEXT,'YYYYMM') <= TO_DATE('202312','YYYYMM')
AND from_district in {create_top_five_str(df,"from_district")}
GROUP BY from_district,date
ORDER BY from_district, date
'''

df_2 = pd.read_sql_query(monthly_outflow_of_top_five_outflow,engine)
display(df_2)

,from_district,date,total_outflow
0,강남구,202301,3843
1,강남구,202302,5853
2,강남구,202303,7491
3,강남구,202304,5523
4,강남구,202305,5256
5,강남구,202306,4675
6,강남구,202307,5362
7,강남구,202308,6321
8,강남구,202309,4228
9,강남구,202310,5194
